# Team agents: subagent steps in the progress display

A team agent delegates work to subagents through `delegate_task`. Each `delegate_task` step now carries:

- `execution_id`: the subagent's own platform execution (`null` for a subagent that ran in-process)
- `steps`: the subagent's own steps, in the same format, nested under the delegation

The SDK's progress display (`agent.run(..., progress_format="logs" | "status")`) used to show only top-level steps, so a delegation was one opaque line. This notebook shows the new behaviour from branch `feature/ENG-3708-sdk-nested-delegation-progress` ([aixplain/aiXplain#1153](https://github.com/aixplain/aiXplain/pull/1153)).

The engine side shipped in aixplain-agents [#439](https://github.com/aixplain/aixplain-agents/pull/439).

The notebook has two parts:

1. **A real run**: `team.run(..., progress_format="logs")` against a live team agent with two subagents (a web researcher that uses Tavily Web Search, and a haiku writer). This needs `AIXPLAIN_API_KEY` and `TEAM_AGENT_ID`, plus `BACKEND_URL` / `MODELS_RUN_URL` for a non-prod environment.
2. **Replays**: the step tree of a recorded team run (whose first Tavily call fails and is retried), fed through the same tracker `agent.run` uses, in every display mode. No API key is needed for this part.

In [1]:
# Install the SDK from the PR branch (dependencies first, then this branch's build over any installed release).
%pip install -q "aiXplain @ git+https://github.com/aixplain/aiXplain.git@feature/ENG-3708-sdk-nested-delegation-progress"
%pip install -q --force-reinstall --no-deps "aiXplain @ git+https://github.com/aixplain/aiXplain.git@feature/ENG-3708-sdk-nested-delegation-progress"


If the import below still points at an older version, restart the kernel once after the install.

In [2]:
from importlib.metadata import version

from aixplain.v2 import agent_progress
from aixplain.v2.agent_progress import AgentProgressTracker, ProgressFormat

assert hasattr(agent_progress, "_is_delegation"), "this SDK build predates nested progress: restart the kernel and rerun"
print("aiXplain SDK", version("aiXplain"), "with nested delegation progress")


aiXplain SDK 0.3.0 with nested delegation progress


## A real run

A team agent whose two subagents are deployed assets, run with `progress_format="logs"` and `progress_verbosity=2`. Each delegation prints a `▸` header with the subagent's platform `execution_id`, then the subagent's own steps (`5.1`, `5.2`, …) as they finish, then its ✓ line. One line at the bottom stays live: the step running right now.

In [3]:
import os

if os.getenv("AIXPLAIN_API_KEY") and os.getenv("TEAM_AGENT_ID"):
    from aixplain import Aixplain

    hosts = {key: os.environ[env] for key, env in (("backend_url", "BACKEND_URL"), ("model_url", "MODELS_RUN_URL")) if os.getenv(env)}
    aix = Aixplain(api_key=os.environ["AIXPLAIN_API_KEY"], **hosts)
    team = aix.Agent.get(os.environ["TEAM_AGENT_ID"])
    result = team.run("Latest news about the Mediterranean Sea", progress_format="logs", progress_verbosity=2, timeout=600)
    print("\n" + result.data.output)
else:
    print("Set AIXPLAIN_API_KEY and TEAM_AGENT_ID (and BACKEND_URL / MODELS_RUN_URL off prod) to run a live team agent.")


✓ Step  1 · ⏱ 00:01.57 · API  1 · $0.000166 · ↓816 ↑73 (889) · e2e-3708-news-team ⧈ GPT-4o Mini
▸ Step  2 · e2e-3708-news-team ⚒ delegate_task → web researcher
✓ Step  2 · ⏱ 00:00.00 · API  1 · $0.000000 · e2e-3708-news-team ⚒ delegate_task → web researcher
▸ Step  3 · e2e-3708-news-team ⚒ delegate_task → haiku writer
✓ Step  3 · ⏱ 00:00.00 · API  1 · $0.000000 · e2e-3708-news-team ⚒ delegate_task → haiku writer
✓ Step  4 · ⏱ 00:00.00 · API  1 · $0.000199 · ↓975 ↑88 (1063) · e2e-3708-news-team ⧈ GPT-4o Mini
▸ Step  5 · e2e-3708-news-team ⚒ delegate_task → e2e-3708-web-researcher
▸ Step  6 · e2e-3708-news-team ⚒ delegate_task → e2e-3708-haiku-writer
  ✓ Step 6.1 · ⏱ 00:00.00 · API  1 · $0.000123 · ↓745 ↑18 (763) · e2e-3708-haiku-writer ⧈ GPT-4o Mini
✓ Step  6 · ⏱ 00:06.11 · API  1 · $0.000147 · e2e-3708-news-team ⚒ delegate_task → e2e-3708-haiku-writer
  ✓ Step 5.1 · ⏱ 00:00.00 · API  1 · $0.000138 · ↓853 ↑17 (870) · e2e-3708-web-researcher ⧈ GPT-4o Mini
  ✓ Step 5.2 · ⏱ 00:00.00 · API 

## Replays: every display mode

The rest of the notebook needs no API key. It replays a recorded team run through the tracker poll by poll.

## The recorded run

Four top-level steps: the coordinator plans, two `delegate_task` steps (each with its subagent's steps nested under `steps`), and the coordinator's final answer.

In [4]:
STEPS = [{'step_id': 'llm-d0e4a12e05a5',
  'status': 'completed',
  'agent': {'id': 'e2e-3708-team', 'name': 'e2e_3708_team', 'is_system_agent': False},
  'unit': {'id': '669a63646eb56306647e1091',
           'name': 'GPT-4o Mini',
           'display_name': None,
           'path': 'openai/gpt-4o-mini/openai',
           'type': 'model',
           'action': None},
  'start_time': '2026-10-01T11:36:32.604550+00:00',
  'end_time': '2026-10-01T11:36:34.775632+00:00',
  'run_time': 2.1801631450653076,
  'thought': None,
  'task': None,
  'action': None,
  'input': 'Latest news about the Mediterranean Sea',
  'output': None,
  'input_tokens': 561,
  'output_tokens': 76,
  'total_tokens': 637,
  'used_credits': 0.00012974999999999998,
  'api_calls': 1,
  'diagnostic_error_codes': []},
 {'step_id': 'call_BhzP61VNMCAXMibgO7Tgbyzq',
  'status': 'completed',
  'agent': {'id': 'e2e-3708-team', 'name': 'e2e_3708_team', 'is_system_agent': False},
  'unit': {'id': None, 'name': 'delegate_task', 'display_name': None, 'path': None, 'type': 'tool', 'action': None},
  'start_time': '2026-10-01T11:36:34.776025+00:00',
  'end_time': '2026-10-01T11:36:50.278973+00:00',
  'run_time': 15.493623971939087,
  'thought': None,
  'task': None,
  'action': None,
  'input': '{"agent_name": "web_researcher", "task": "Find the latest news articles about the Mediterranean Sea."}',
  'output': 'Here are the latest news articles about the Mediterranean Sea:\n'
            '\n'
            '- **Pope urges Europe to do more for migrants as he visits gateway island**: The Pope paid tribute to '
            'migrants who lost their lives at sea during a visit to …',
  'input_tokens': None,
  'output_tokens': None,
  'total_tokens': None,
  'used_credits': 0.0019751400000000002,
  'api_calls': 1,
  'diagnostic_error_codes': [],
  'execution_id': '949-09e678c5-1840-4c2f-9e2d-0b3a2b332bb1',
  'steps': [{'step_id': 'llm-f0e06ccff342',
             'status': 'completed',
             'agent': {'id': '6abe3eb0a6dca5a6c818d6a7', 'name': 'e2e-3708-web-researcher', 'is_system_agent': False},
             'unit': {'id': '669a63646eb56306647e1091',
                      'name': 'GPT-4o Mini',
                      'display_name': None,
                      'path': 'openai/gpt-4o-mini/openai',
                      'type': 'model',
                      'action': None},
             'start_time': '2026-10-01T11:36:41.570026+00:00',
             'end_time': '2026-10-01T11:36:42.680309+00:00',
             'run_time': 1.7914161682128906,
             'thought': None,
             'task': None,
             'action': None,
             'input': 'Find the latest news articles about the Mediterranean Sea.',
             'output': None,
             'input_tokens': 867,
             'output_tokens': 52,
             'total_tokens': 919,
             'used_credits': 0.00016125,
             'api_calls': 1,
             'diagnostic_error_codes': []},
            {'step_id': 'call_4q971nL6bJk4J9ySNRPFdUz3',
             'status': 'failed',
             'agent': {'id': '6abe3eb0a6dca5a6c818d6a7', 'name': 'e2e-3708-web-researcher', 'is_system_agent': False},
             'unit': {'id': '6931bdf462eb386b7158def3',
                      'name': 'Tavily Web Search',
                      'display_name': 'Tavily Web Search',
                      'path': 'tavily/tavily-web-search/tavily',
                      'type': 'tool',
                      'action': {'name': 'search', 'description': None}},
             'start_time': '2026-10-01T11:36:42.681274+00:00',
             'end_time': '2026-10-01T11:36:42.804253+00:00',
             'run_time': None,
             'thought': None,
             'task': None,
             'action': None,
             'input': '{"num_results": "5", "query": "Mediterranean Sea news", "include_domains": [{"domain": '
                      '"bbc.com"}, {"domain": "cnn.com"}, {"domain": "reuters.com"}, {"domain": "aljazeera.com"}, '
                      '{"domain": "theguardian.com"}]}',
             'output': "ERROR: Action 'search' on 'Tavily Web Search' failed: Operation failed: "
                       '{"detail":[{"type":"string_type","loc":["body","include_domains",0],"msg":"Input should be a '
                       'valid string","input":{"domain":"bbc.com"}},{"type":"st …',
             'input_tokens': None,
             'output_tokens': None,
             'total_tokens': None,
             'used_credits': None,
             'api_calls': 1,
             'diagnostic_error_codes': ['TOOL_FAILED']},
            {'step_id': 'llm-1f4424d045db',
             'status': 'completed',
             'agent': {'id': '6abe3eb0a6dca5a6c818d6a7', 'name': 'e2e-3708-web-researcher', 'is_system_agent': False},
             'unit': {'id': '669a63646eb56306647e1091',
                      'name': 'GPT-4o Mini',
                      'display_name': None,
                      'path': 'openai/gpt-4o-mini/openai',
                      'type': 'model',
                      'action': None},
             'start_time': '2026-10-01T11:36:42.832877+00:00',
             'end_time': '2026-10-01T11:36:43.767589+00:00',
             'run_time': 0.9621448516845703,
             'thought': None,
             'task': None,
             'action': None,
             'input': 'Find the latest news articles about the Mediterranean Sea.',
             'output': None,
             'input_tokens': 1148,
             'output_tokens': 41,
             'total_tokens': 1189,
             'used_credits': 0.00019679999999999999,
             'api_calls': 1,
             'diagnostic_error_codes': []},
            {'step_id': 'call_1bSnCYo2gI8bu0jUrjj5QtDo',
             'status': 'completed',
             'agent': {'id': '6abe3eb0a6dca5a6c818d6a7', 'name': 'e2e-3708-web-researcher', 'is_system_agent': False},
             'unit': {'id': '6931bdf462eb386b7158def3',
                      'name': 'Tavily Web Search',
                      'display_name': 'Tavily Web Search',
                      'path': 'tavily/tavily-web-search/tavily',
                      'type': 'tool',
                      'action': {'name': 'search', 'description': None}},
             'start_time': '2026-10-01T11:36:43.768404+00:00',
             'end_time': '2026-10-01T11:36:45.911772+00:00',
             'run_time': 2.0905354022979736,
             'thought': None,
             'task': None,
             'action': None,
             'input': '{"num_results": "5", "query": "Mediterranean Sea news", "include_domains": ["bbc.com", '
                      '"cnn.com", "reuters.com", "aljazeera.com", "theguardian.com"]}',
             'output': '{"message": "Action \'search\' on \'Tavily Web Search\' succeeded.", "data": {"details": '
                       '[{"data": "# Mediterranean Sea\\n\\nPope Leo on a visit to Lampedusa, Italy on July 4, '
                       '2026\\n\\n## Pope urges Europe to do more for migran …',
             'input_tokens': None,
             'output_tokens': None,
             'total_tokens': None,
             'used_credits': 0.00075,
             'api_calls': 1,
             'diagnostic_error_codes': []},
            {'step_id': 'llm-1aebc4afd121',
             'status': 'completed',
             'agent': {'id': '6abe3eb0a6dca5a6c818d6a7', 'name': 'e2e-3708-web-researcher', 'is_system_agent': False},
             'unit': {'id': '669a63646eb56306647e1091',
                      'name': 'GPT-4o Mini',
                      'display_name': None,
                      'path': 'openai/gpt-4o-mini/openai',
                      'type': 'model',
                      'action': None},
             'start_time': '2026-10-01T11:36:45.994583+00:00',
             'end_time': '2026-10-01T11:36:48.282251+00:00',
             'run_time': 2.368774652481079,
             'thought': None,
             'task': None,
             'action': None,
             'input': 'Find the latest news articles about the Mediterranean Sea.',
             'output': 'Here are the latest news articles about the Mediterranean Sea:\n'
                       '\n'
                       '- **Pope urges Europe to do more for migrants as he visits gateway island**: The Pope paid '
                       'tribute to migrants who lost their lives at sea during a visit to …',
             'input_tokens': 2754,
             'output_tokens': 208,
             'total_tokens': 2962,
             'used_credits': 0.0005379,
             'api_calls': 1,
             'diagnostic_error_codes': []}]},
 {'step_id': 'call_sYE0Wk3M0fOOVZb7fbR1vcMJ',
  'status': 'completed',
  'agent': {'id': 'e2e-3708-team', 'name': 'e2e_3708_team', 'is_system_agent': False},
  'unit': {'id': None, 'name': 'delegate_task', 'display_name': None, 'path': None, 'type': 'tool', 'action': None},
  'start_time': '2026-10-01T11:36:34.776219+00:00',
  'end_time': '2026-10-01T11:36:39.530457+00:00',
  'run_time': 4.753577947616577,
  'thought': None,
  'task': None,
  'action': None,
  'input': '{"agent_name": "haiku_writer", "task": "Write a haiku about the Mediterranean Sea."}',
  'output': 'Azure waves whisper,  \nSunset paints the horizon,  \nAncient shores embrace.',
  'input_tokens': None,
  'output_tokens': None,
  'total_tokens': None,
  'used_credits': 0.0001494,
  'api_calls': 1,
  'diagnostic_error_codes': [],
  'execution_id': '949-198b8e76-9e27-43e4-8353-7fee147bd31b',
  'steps': [{'step_id': 'llm-861a49da3777',
             'status': 'completed',
             'agent': {'id': '6abbcb9abf00785c9198bec2', 'name': 'e2e-3708-haiku-writer', 'is_system_agent': False},
             'unit': {'id': '669a63646eb56306647e1091',
                      'name': 'GPT-4o Mini',
                      'display_name': None,
                      'path': 'openai/gpt-4o-mini/openai',
                      'type': 'model',
                      'action': None},
             'start_time': '2026-10-01T11:36:37.840101+00:00',
             'end_time': '2026-10-01T11:36:38.468050+00:00',
             'run_time': 1.1315371990203857,
             'thought': None,
             'task': None,
             'action': None,
             'input': 'Write a haiku about the Mediterranean Sea.',
             'output': 'Azure waves whisper,  \nSunset paints the horizon,  \nAncient shores embrace.',
             'input_tokens': 758,
             'output_tokens': 18,
             'total_tokens': 776,
             'used_credits': 0.0001245,
             'api_calls': 1,
             'diagnostic_error_codes': []}]},
 {'step_id': 'llm-e7f73dbe4ae2',
  'status': 'completed',
  'agent': {'id': 'e2e-3708-team', 'name': 'e2e_3708_team', 'is_system_agent': False},
  'unit': {'id': '669a63646eb56306647e1091',
           'name': 'GPT-4o Mini',
           'display_name': None,
           'path': 'openai/gpt-4o-mini/openai',
           'type': 'model',
           'action': None},
  'start_time': '2026-10-01T11:36:50.299787+00:00',
  'end_time': '2026-10-01T11:36:52.882366+00:00',
  'run_time': 2.6024529933929443,
  'thought': None,
  'task': None,
  'action': None,
  'input': 'Latest news about the Mediterranean Sea',
  'output': 'Here are the latest news articles about the Mediterranean Sea:\n'
            '\n'
            '- **Pope urges Europe to do more for migrants as he visits gateway island**: The Pope paid tribute to '
            'migrants who lost their lives at sea during a visit to …',
  'input_tokens': 888,
  'output_tokens': 238,
  'total_tokens': 1126,
  'used_credits': 0.000276,
  'api_calls': 1,
  'diagnostic_error_codes': []}]

for i, step in enumerate(STEPS, 1):
    nested = step.get("steps")
    print(f"Step {i}: {step['unit']['name']:<14} nested steps: {len(nested) if nested is not None else '-'}"
          f"   execution_id: {step.get('execution_id') or '-'}")


Step 1: GPT-4o Mini    nested steps: -   execution_id: -
Step 2: delegate_task  nested steps: 5   execution_id: 949-09e678c5-1840-4c2f-9e2d-0b3a2b332bb1
Step 3: delegate_task  nested steps: 1   execution_id: 949-198b8e76-9e27-43e4-8353-7fee147bd31b
Step 4: GPT-4o Mini    nested steps: -   execution_id: -


## Replaying a run through the tracker

`agent.run(query, progress_format=..., progress_verbosity=...)` creates an `AgentProgressTracker`, calls `update()` with every poll response, and `finish()`es it. `replay` does the same with recorded snapshots, so what you see is exactly what `agent.run` prints.

In [5]:
import copy
import time
from types import SimpleNamespace


def _poll(steps):
    return SimpleNamespace(to_dict=lambda: {"data": {"steps": steps}})


def replay(snapshots, fmt="logs", verbosity=1, delay=0.0):
    tracker = AgentProgressTracker(poll_func=lambda _: None)
    tracker.start(format=ProgressFormat(fmt), verbosity=verbosity)
    for snapshot in snapshots:
        tracker.update(_poll(snapshot))
        time.sleep(delay)
    tracker.finish(SimpleNamespace(status="SUCCESS", to_dict=lambda: {"data": {"steps": snapshots[-1]}}))


def in_progress(n_research, n_haiku):
    """The run part-way through: both delegations running, with this many subagent steps each."""
    snapshot = copy.deepcopy(STEPS[:3])
    for delegation, n in ((snapshot[1], n_research), (snapshot[2], n_haiku)):
        delegation["steps"] = delegation["steps"][:n]
        delegation.update(status="executing", output=None)
        if delegation["steps"]:
            delegation["steps"][-1].update(status="executing", output=None)
    return snapshot


# The finished run, poll by poll: both subagents working in parallel, then done.
LIVE = [in_progress(1, 0), in_progress(1, 1), in_progress(2, 1), in_progress(4, 1), in_progress(5, 1), STEPS]


## `logs`: the finished run

Each delegation gets a `▸` header, then its subagent's steps indented under it (`2.1`, `2.2`, …), then its own ✓ line. The failed Tavily call shows as ✗, and the retry follows it. The completion total is the sum of the four top-level steps: a delegation carries its subagent's total cost, so nested steps aren't counted twice.

In [6]:
replay([STEPS], "logs", verbosity=1)

✓ Step  1 · ⏱ 00:00.00 · API  1 · $0.000130 · ↓561 ↑76 (637) · e2e_3708_team ⧈ GPT-4o Mini
▸ Step  2 · e2e_3708_team ⚒ delegate_task → web_researcher
  ✓ Step 2.1 · ⏱ 00:00.00 · API  1 · $0.000161 · ↓867 ↑52 (919) · e2e-3708-web-researcher ⧈ GPT-4o Mini
  ✗ Step 2.2 · ⏱ 00:00.00 · API  1 · $0.000000 · e2e-3708-web-researcher ⚒ Tavily Web Search
  ✓ Step 2.3 · ⏱ 00:00.00 · API  1 · $0.000197 · ↓1148 ↑41 (1189) · e2e-3708-web-researcher ⧈ GPT-4o Mini
  ✓ Step 2.4 · ⏱ 00:00.00 · API  1 · $0.000750 · e2e-3708-web-researcher ⚒ Tavily Web Search
  ✓ Step 2.5 · ⏱ 00:00.00 · API  1 · $0.000538 · ↓2754 ↑208 (2962) · e2e-3708-web-researcher ⧈ GPT-4o Mini
✓ Step  2 · ⏱ 00:00.00 · API  1 · $0.001975 · e2e_3708_team ⚒ delegate_task → web_researcher
▸ Step  3 · e2e_3708_team ⚒ delegate_task → haiku_writer
  ✓ Step 3.1 · ⏱ 00:00.00 · API  1 · $0.000124 · ↓758 ↑18 (776) · e2e-3708-haiku-writer ⧈ GPT-4o Mini
✓ Step  3 · ⏱ 00:00.00 · API  1 · $0.000149 · e2e_3708_team ⚒ delegate_task → haiku_writer
✓ St

## `logs`, live

The same run as it arrives, poll by poll. Exactly one line stays live at the bottom: the step that is running now, followed into a running delegation. With parallel delegations the log is in the order things finish, and each line's number (`2.3`, `3.1`) says which delegation it belongs to.

In [7]:
replay(LIVE, "logs", verbosity=1, delay=1.0)

✓ Step  1 · ⏱ 00:00.00 · API  1 · $0.000130 · ↓561 ↑76 (637) · e2e_3708_team ⧈ GPT-4o Mini
▸ Step  2 · e2e_3708_team ⚒ delegate_task → web_researcher
▸ Step  3 · e2e_3708_team ⚒ delegate_task → haiku_writer
  ✓ Step 2.1 · ⏱ 00:02.00 · API  1 · $0.000161 · ↓867 ↑52 (919) · e2e-3708-web-researcher ⧈ GPT-4o Mini
  ✗ Step 2.2 · ⏱ 00:01.00 · API  1 · $0.000000 · e2e-3708-web-researcher ⚒ Tavily Web Search
  ✓ Step 2.3 · ⏱ 00:00.00 · API  1 · $0.000197 · ↓1148 ↑41 (1189) · e2e-3708-web-researcher ⧈ GPT-4o Mini
  ✓ Step 2.4 · ⏱ 00:01.00 · API  1 · $0.000750 · e2e-3708-web-researcher ⚒ Tavily Web Search
  ✓ Step 2.5 · ⏱ 00:01.00 · API  1 · $0.000538 · ↓2754 ↑208 (2962) · e2e-3708-web-researcher ⧈ GPT-4o Mini
✓ Step  2 · ⏱ 00:05.03 · API  1 · $0.001975 · e2e_3708_team ⚒ delegate_task → web_researcher
  ✓ Step 3.1 · ⏱ 00:04.02 · API  1 · $0.000124 · ↓758 ↑18 (776) · e2e-3708-haiku-writer ⧈ GPT-4o Mini
✓ Step  3 · ⏱ 00:05.03 · API  1 · $0.000149 · e2e_3708_team ⚒ delegate_task → haiku_writer
✓ St

## `logs`, `progress_verbosity=2` and `3`

From verbosity 2, the delegation header shows the subagent's `execution_id`. You can open or poll that execution on its own. Verbosity 3 adds each step's input and output, indented under the step.

In [8]:
replay([STEPS], "logs", verbosity=2)

✓ Step  1 · ⏱ 00:00.00 · API  1 · $0.000130 · ↓561 ↑76 (637) · e2e_3708_team ⧈ GPT-4o Mini
▸ Step  2 · e2e_3708_team ⚒ delegate_task → web_researcher · ↳ execution 949-09e678c5-1840-4c2f-9e2d-0b3a2b332bb1
  ✓ Step 2.1 · ⏱ 00:00.00 · API  1 · $0.000161 · ↓867 ↑52 (919) · e2e-3708-web-researcher ⧈ GPT-4o Mini
  ✗ Step 2.2 · ⏱ 00:00.00 · API  1 · $0.000000 · e2e-3708-web-researcher ⚒ Tavily Web Search
  ✓ Step 2.3 · ⏱ 00:00.00 · API  1 · $0.000197 · ↓1148 ↑41 (1189) · e2e-3708-web-researcher ⧈ GPT-4o Mini
  ✓ Step 2.4 · ⏱ 00:00.00 · API  1 · $0.000750 · e2e-3708-web-researcher ⚒ Tavily Web Search
  ✓ Step 2.5 · ⏱ 00:00.00 · API  1 · $0.000538 · ↓2754 ↑208 (2962) · e2e-3708-web-researcher ⧈ GPT-4o Mini
✓ Step  2 · ⏱ 00:00.00 · API  1 · $0.001975 · e2e_3708_team ⚒ delegate_task → web_researcher
▸ Step  3 · e2e_3708_team ⚒ delegate_task → haiku_writer · ↳ execution 949-198b8e76-9e27-43e4-8353-7fee147bd31b
  ✓ Step 3.1 · ⏱ 00:00.00 · API  1 · $0.000124 · ↓758 ↑18 (776) · e2e-3708-haiku-writer

In [9]:
replay([STEPS], "logs", verbosity=3)

✓ Step  1 · ⏱ 00:00.00 · API  1 · $0.000130 · ↓561 ↑76 (637) · e2e_3708_team ⧈ GPT-4o Mini
  ← Query
    │ Latest news about the Mediterranean Sea
▸ Step  2 · e2e_3708_team ⚒ delegate_task → web_researcher · ↳ execution 949-09e678c5-1840-4c2f-9e2d-0b3a2b332bb1
  ✓ Step 2.1 · ⏱ 00:00.00 · API  1 · $0.000161 · ↓867 ↑52 (919) · e2e-3708-web-researcher ⧈ GPT-4o Mini
    ← Context
      │ Find the latest news articles about the Mediterranean Sea.
  ✗ Step 2.2 · ⏱ 00:00.00 · API  1 · $0.000000 · e2e-3708-web-researcher ⚒ Tavily Web Search
    ← Input
      │ {
    │   "num_results": "5",
    │   "query": "Mediterranean Sea news",
    │   "include_domains": [
    │     {
    │       "domain": "bbc.com"
    │     },
    │     {
    │       "domain": "cnn.com"
    │     },
    │     {
    │       "domain": "reuters.com"
    │     },
    │     {
    │       "domain": "aljazeera.com"
    │     },
    │     {
    │       "domain": "theguardian.com"
    │     }
    │   ]
    │ }
    → Result ✓
    

## `status`: one updating line

The single line follows a running delegation down to the step its subagent is on, e.g. `Step 2.4 · … · e2e-3708-web-researcher ⚒ Tavily Web Search`. With parallel delegations it stays on the one that is still running.

In [10]:
replay(LIVE, "status", verbosity=1, delay=1.0)

✓ Step  4 · ⏱ 00:00.00 · API  1 · $0.000276 · ↓888 ↑238 (1126) · e2e_3708_team ⧈ GPT-4o Mini
✓ Completed 4 steps · ⏱ 00:06.02 · API 4 · $0.002530 · ↓1449 ↑314 (1763)
